# Ćwiczenia 2: Model ML, FastAPI i serwowanie predykcji

*Studia zaoczne | 1,5h*

## Cel

- Wytrenowanie modelu detekcji fraudów,
- Serwowanie modelu przez FastAPI,
- Podłączenie do Kafki — scoring w czasie rzeczywistym.

---

## Część 1: Trening modelu (25 min)

### Zadanie 1.1 — Przygotuj dane i wytrenuj model

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, 2000).clip(5, 5000),
    'hour': np.random.normal(14, 4, 2000).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, 2000),
    'tx_per_day': np.random.poisson(3, 2000),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, 100),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], 100),
    'is_electronics': np.random.binomial(1, 0.7, 100),
    'tx_per_day': np.random.poisson(8, 100),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


### Zadanie 1.2 — Podziel dane, wytrenuj, oceń

In [ ]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, digits=4))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model zapisany do fraud_model.pkl")

---

## Część 2: FastAPI (25 min)

### Zadanie 2.1 — Utwórz API serwujące model

In [ ]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

class ScoringResult(BaseModel):
    is_fraud: bool
    fraud_probability: float

@app.post('/score', response_model=ScoringResult)
def score(transaction: Transaction):
    features = np.array([[transaction.amount, transaction.hour,
                          transaction.is_electronics, transaction.tx_per_day]])
    prob = float(model.predict_proba(features)[0][1])
    return {"is_fraud": prob >= 0.5, "fraud_probability": prob}

@app.get('/health')
def health():
    return {"status": "ok"}


Uruchom w terminalu:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

### Zadanie 2.2 — Przetestuj API

In [ ]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

r2 = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "hour": 3, "is_electronics": 1, "tx_per_day": 12})
print("Podejrzana:", r2.json())

---

## Część 3: Kafka + ML (25 min)

### Zadanie 3.1 — Konsument z scoringiem ML

Napisz konsumenta, który czyta z `transactions`, odpytuje API i wysyła alerty.

In [ ]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json
import requests

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value
    tx_id = tx.get('tx_id')
    amount = float(tx.get('amount', 0.0))
    is_electronics = int(tx.get('is_electronics', 0))
    tx_per_day = int(tx.get('tx_per_day', 5))

    if 'hour' in tx:
        hour = int(tx['hour'])
    else:
        timestamp = tx.get('timestamp')
        hour = datetime.fromisoformat(timestamp).hour if timestamp else 0

    payload = {
        'amount': amount,
        'hour': hour,
        'is_electronics': is_electronics,
        'tx_per_day': tx_per_day
    }

    response = requests.post(API_URL, json=payload)
    result = response.json()

    if result.get('is_fraud'):
        alert = {
            'tx_id': tx_id,
            'amount': amount,
            'fraud_probability': result.get('fraud_probability'),
            'timestamp': tx.get('timestamp'),
            'alert_time': datetime.utcnow().isoformat() + 'Z'
        }
        alert_producer.send('alerts', value=alert)
        alert_producer.flush()
        print(f"ALERT: {tx_id} -> {alert['fraud_probability']:.2f}")
    else:
        print(f"OK: {tx_id} -> {result.get('fraud_probability'):.2f}")


### Zadanie 3.2 — Uruchom pipeline

W 3 terminalach:
1. `python producer.py` (z Ćw. 1)
2. `uvicorn fraud_api:app --host 0.0.0.0 --port 8001`
3. `python ml_consumer.py`

Obserwuj alerty.

---

## Praca domowa

1. Porównaj wyniki scoringu regułowego (Ćw. 1) vs ML — który lepiej wykrywa?
2. Dodaj endpoint `GET /health` do API.
3. Wypchnij do Git.

**Następne zajęcia:** Apache Spark — przetwarzanie na dużą skalę.